# DP2 AnaCal: cluster shear response per stack and radial bin

Reads the outputs of `pipeline.yml` in this folder (`CLClusterShearCalibrator`) and compares the
tangential response $\langle R_T \rangle$ in each cluster stack and radial bin with the global AnaCal response
of the source sample.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

OUTDIR = "/pscratch/sd/e/edujb/dp2/outputs/anacal"
MIN_PAIRS = 1000  # radial bins with fewer source pairs are drawn hollow

## Global response and per-stack response

In [ ]:
with h5py.File(f"{OUTDIR}/shear_tomography_catalog.hdf5") as f:
    global_R = float(np.atleast_1d(f["response/R"][()])[0])

stacks = []
with h5py.File(f"{OUTDIR}/cluster_shear_response.hdf5") as f:
    cal = f["calibration"]
    print(dict(cal.attrs))
    edges = cal.attrs["distance_bins"]
    for name in sorted(cal):
        g = cal[name]
        if "R_T" in g:
            err = g["R_T_err"][:] if "R_T_err" in g else np.full(g["R_T"].shape, np.nan)
            stacks.append((name, dict(g.attrs), g["R_T"][:], err))
r = np.sqrt(edges[:-1] * edges[1:])

print(f"global R = {global_R:.4f}")
for name, attrs, R_T, err in stacks:
    print(name, attrs["n_clusters"], "clusters")
    for k in range(len(r)):
        print(f"   R = {r[k]:.2f} Mpc   R_T = {R_T[k]:.4f} +- {err[k]:.4f}   pairs = {attrs['n_sources_per_radial_bin'][k]}")

## Plot

In [ ]:
fig, axes = plt.subplots(1, len(stacks), figsize=(5 * len(stacks), 5.6), sharey=True, squeeze=False)
axes = axes[0]
color = "#2a78d6"
for ax, (name, attrs, R_T, err) in zip(axes, stacks):
    n_pairs = np.asarray(attrs["n_sources_per_radial_bin"])
    few = n_pairs < MIN_PAIRS
    ax.plot(r, R_T, color=color, lw=2)
    ax.errorbar(r, R_T, yerr=err, fmt="none", ecolor=color, elinewidth=1.5, capsize=3)
    ax.plot(r[~few], R_T[~few], "o", color=color, ms=7)
    ax.plot(r[few], R_T[few], "o", mfc="white", mec=color, mew=2, ms=7)
    ax.axhline(global_R, color="0.4", ls="--", lw=1.5)
    for x, y, n in zip(r, R_T, n_pairs):
        ax.annotate(f"{n}", (x, y), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=7, color="0.35")
    ax.set_title(f"{attrs['z_min']:.1f} < z < {attrs['z_max']:.1f},  {attrs['rich_min']:.0f} < λ < {attrs['rich_max']:.0f}\n"
                 f"{attrs['n_clusters']} clusters", fontsize=10)
    ax.set_xscale("log")
    ax.set_xticks([0.3, 1, 3], ["0.3", "1", "3"])
    ax.set_xlabel("R [Mpc]")
    ax.grid(alpha=0.25, lw=0.5)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel(r"$\langle R_T \rangle$")
axes[-1].text(edges[-1], global_R - 0.004, f"global R = {global_R:.3f}", ha="right", va="top", fontsize=8, color="0.3")
fig.suptitle(f"DP2 AnaCal: tangential response per radial bin (errors: bootstrap over clusters; numbers: source pairs; hollow: < {MIN_PAIRS} pairs)", fontsize=10)
#fig.tight_layout()